<a href="https://colab.research.google.com/github/tor-apiwit/Option-Pricing-Greeks-Calculator/blob/main/Option_Pricing_%26_Greeks_Calculator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import norm
import yfinance as yf

In [ ]:
def black_scholes_price(S, K, T, r, sigma, option_type="call"):

  """
    Calculates the theoretical price of a European option
    using the Black-Scholes-Merton model.

    Parameters:
    -----------
    S : float
        Spot price of the underlying asset.
    K : float
        Strike price of the option contract.
    T : float
        Time to maturity in years (e.g., 0.5 for 6 months).
    r : float
        Annualized continuously compounded risk-free interest rate.
    sigma : float
        Annualized volatility of the underlying asset's returns.
    option_type : str, optional
        The type of option contract: "call" or "put" (default is "call").

    Raises:
    -------
    ValueError
        If option_type is not 'call' or 'put'.
    """
  # --- 1. Calculate Probability Factors (d1 and d2) ---
  # d1 measures how far in-the-money the option is, adjusted for volatility and time.
  d1 = (np.log(S / K) + (r + 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))
  d2 = d1 - sigma * np.sqrt(T)

  # --- 2. Conditional Pricing and Greeks Calculation ---
  if option_type == "call":
    # Standard Black-Scholes Call Pricing Formula
    price = S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)

  elif option_type == "put":
      # Standard Black-Scholes Put Pricing Formula
      price = K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)

  else:
      raise ValueError("option_type must be 'call' or 'put'")

  return price

In [ ]:
def monte_carlo_price(S_0, K, T, r, sigma, option_type="call"):
    """
    Prices a European option (Call or Put) using the Monte Carlo simulation method.

    This function simulates terminal stock prices under a risk-neutral measure
    based on Geometric Brownian Motion (GBM), evaluates the payoff scenarios,
    and discounts the expected payoff back to present value.

    Parameters:
    -----------
    S_0 : float
        Current asset price (Spot price at t=0).
    K : float
        Strike price of the option contract.
    T : float
        Time to maturity in years (e.g., 0.5 for 6 months).
    r : float
        Annualized continuously compounded risk-free interest rate.
    sigma : float
        Annualized volatility of the underlying asset's returns.
    option_type : str, optional
        The type of option to price. Must be either 'call' or 'put' (default is 'call').

    Returns:
    --------
    float
        The estimated fair value (present value) of the option.

    Raises:
    -------
    ValueError
        If the 'option_type' provided is not 'call' or 'put'.

    Example:
    --------
    >>> price = monte_carlo_price(100, 100, 1, 0.05, 0.2, option_type="call")
    """
    # Number of random paths to simulate for accuracy
    n_sims = 500_000

    # Generate standard normal random variables (Z ~ N(0,1))
    z_independent = np.random.standard_normal(n_sims)

    # --- 1. Simulate Terminal Stock Prices (GBM) ---
    # Calculates the log-returns according to Geometric Brownian Motion
    returns = (r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * z_independent
    # Projects the final stock prices at maturity (T)
    S_T = S_0 * np.exp(returns)

    # --- 2. Evaluate Option Payoffs at Maturity ---
    if option_type == "call":
        # Call Payoff = Max(S_T - K, 0)
        payout_scenarios = np.maximum(S_T - K, 0)

    elif option_type == "put":
        # Put Payoff = Max(K - S_T, 0)
        payout_scenarios = np.maximum(K - S_T, 0)

    else:
        raise ValueError("option_type must be 'call' or 'put'")

    # --- 3. Expectation & Discounting ---
    # Find the average payoff across all 500,000 simulated paths
    expected_payout_future = payout_scenarios.mean()

    # Discount the expected future payoff back to present value using e^(-rT)
    price = expected_payout_future * np.exp(-T * r)

    return price

In [ ]:
def binomial_tree_price(S_0, K, T, r, sigma, option_type="call"):
    """
    Calculates the theoretical price and replication parameters of a European
    option using a 1-step (single-period) Binomial Tree model.

    This function utilizes the no-arbitrage replicating portfolio method
    (combining stock and bonds) to value the option premium.

    Parameters:
    -----------
    S_0 : float
        Current spot price of the underlying asset.
    K : float
        Strike price of the option contract.
    T : float
        Time to maturity in years (representing the length of the single period).
    r : float
        Annualized continuously compounded risk-free interest rate (used for discounting).
    sigma : float
        Annualized volatility of the underlying asset's returns.
    option_type : str, optional
        The type of option contract: "call" or "put" (default is "call").

    Returns:
    --------
    float
        The theoretical fair price (premium) of the option at time t=0.

    Raises:
    -------
    ValueError
        If option_type is not 'call' or 'put'.
    """

    # --- 1. Calculate Up and Down Return Factors ---
    # Computes percentage movements based on asset volatility and time step.
    u = np.exp(sigma * np.sqrt(T))  - 1  # Percentage increase in the "up" state
    d = np.exp(-sigma * np.sqrt(T)) - 1  # Percentage decrease in the "down" state

    # --- 2. Project Future Stock Prices ---
    # Simulates the two possible asset prices at maturity (T).
    S_u = S_0 * (1 + u)
    S_d = S_0 * (1 + d)

    # --- 3. Compute Intrinsic Option Payoffs at Maturity ---
    if option_type == "call":
        payout_u = np.maximum(S_u - K, 0)
        payout_d = np.maximum(S_d - K, 0)

    elif option_type == "put":
        payout_u = np.maximum(K - S_u, 0)
        payout_d = np.maximum(K - S_d, 0)

    else:
        raise ValueError("option_type must be 'call' or 'put'")

    # --- 4. Portfolio Replication & Pricing ---
    # Delta (Δ): The number of shares required in the replicating portfolio to hedge risk.
    delta = (payout_u - payout_d) / (S_0 * (u - d))

    # Bond Values (B): The amount of money borrowed/lent at the risk-free rate.
    # The term (1 + r*T) acts as the linear discount factor for the single period.
    bond_values = (payout_d * (1 + u) - payout_u * (1 + d)) / ((1 + r * T) * (u - d))

    # Replicating Portfolio Price: V_0 = (Δ * S_0) + B
    # By no-arbitrage principles, the option price must equal the cost of this portfolio.
    price = (S_0 * delta) + bond_values

    return price

In [ ]:
def greeks_fdm(model, S_0, K, T, r, sigma, option_type):
    """
    Estimates the option Greeks using the Finite Difference Method (FDM)
    by perturbing inputs via the Model pricing function.

    This function applies numerical differentiation (Central and Central-Second
    Difference schemes) to approximate partial derivatives, serving as a
    validation tool or alternative to exact analytical solutions.

    Parameters:
    -----------
    S_0 : float
        Current spot price of the underlying asset.
    K : float
        Strike price of the option contract.
    T : float
        Time to maturity in years.
    r : float
        Annualized continuously compounded risk-free interest rate.
    sigma : float
        Annualized volatility of the underlying asset's returns.
    option_type : str
        The type of option contract: "call" or "put".

    Returns:
    --------
    tuple of floats
        (delta, gamma, vega, theta, rho) calculated via numerical approximations.
    """

    # --- Define Perturbation Steps (Shocks) ---
    h_S = 0.01 * S_0   # 1% shift in underlying asset price
    h_sigma = 0.01     # 1% absolute shift in volatility (0.01 = 1 percentage point)
    h_r = 0.001        # 0.1% absolute shift in interest rate (10 basis points)

    # =========================================================================
    # 1. Delta & Gamma (Asset Price Sensitivities)
    # =========================================================================
    # Shift stock price up, down, and capture baseline price
    V_S_plus = model(S_0 + h_S, K, T, r, sigma, option_type)
    V_S_minus = model(S_0 - h_S, K, T, r, sigma, option_type)
    V_base = model(S_0, K, T, r, sigma, option_type)

    # Delta (First Derivative): Central Difference Scheme
    delta = (V_S_plus - V_S_minus) / (2 * h_S)

    # Gamma (Second Derivative): Central Second Difference Scheme
    gamma = (V_S_plus - 2 * V_base + V_S_minus) / (h_S ** 2)

    # =========================================================================
    # 2. Vega (Volatility Sensitivity)
    # =========================================================================
    # Shift volatility up and down
    V_sigma_plus = model(S_0, K, T, r, sigma + h_sigma, option_type)
    V_sigma_minus = model(S_0, K, T, r, sigma - h_sigma, option_type)

    # Vega (First Derivative): Central Difference Scheme
    vega = (V_sigma_plus - V_sigma_minus) / (2 * h_sigma)

    # =========================================================================
    # 3. Theta (Time Decay Sensitivity)
    # =========================================================================
    dt = 1 / 252  # Time step representing exactly 1 trading day

    # Ensure there is enough time remaining to step backward and forward
    if T - dt > 0:
        V_t_plus = model(S_0, K, T + dt, r, sigma, option_type)
        V_t_minus = model(S_0, K, T - dt, r, sigma, option_type)

        # Theta (Central Difference): Corrected to accurately capture time decay
        # as maturity decreases (T - dt is later in calendar time than T + dt).
        theta = (V_t_minus - V_t_plus) / (2 * dt)
    else:
        theta = np.nan  # Avoid calculating negative or zero time to maturity

    # =========================================================================
    # 4. Rho (Interest Rate Sensitivity)
    # =========================================================================
    # Shift interest rate up and down
    V_r_plus = model(S_0, K, T, r + h_r, sigma, option_type)
    V_r_minus = model(S_0, K, T, r - h_r, sigma, option_type)

    # Rho (First Derivative): Central Difference Scheme
    rho = (V_r_plus - V_r_minus) / (2 * h_r)

    return delta, gamma, vega, theta, rho

In [ ]:
def greeks_fdm_cf(S, K, T, r, sigma, option_type="call"):

  """
    Calculates the theoretical price and the risk sensitivities (Greeks)
    of a European option using the Black-Scholes-Merton model.

    Parameters:
    -----------
    S : float
        Spot price of the underlying asset.
    K : float
        Strike price of the option contract.
    T : float
        Time to maturity in years (e.g., 0.5 for 6 months).
    r : float
        Annualized continuously compounded risk-free interest rate.
    sigma : float
        Annualized volatility of the underlying asset's returns.
    option_type : str, optional
        The type of option contract: "call" or "put" (default is "call").

    Raises:
    -------
    ValueError
        If option_type is not 'call' or 'put'.
    """
  # --- 1. Calculate Probability Factors (d1 and d2) ---
  # d1 measures how far in-the-money the option is, adjusted for volatility and time.
  d1 = (np.log(S / K) + (r + 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))
  d2 = d1 - sigma * np.sqrt(T)

  # --- 2. Conditional Pricing and Greeks Calculation ---
  if option_type == "call":
    # Call-specific Greeks
    delta = norm.cdf(d1)
    theta = -(S * norm.pdf(d1) * sigma) / (2 * np.sqrt(T)) - r * K * np.exp(-r * T) * norm.cdf(d2)
    rho   = K * T * np.exp(-r * T) * norm.cdf(d2)

  elif option_type == "put":
      # Put-specific Greeks
      delta = norm.cdf(d1) - 1
      theta = -(S * norm.pdf(d1) * sigma) / (2 * np.sqrt(T)) + r * K * np.exp(-r * T) * norm.cdf(-d2)
      rho   = -K * T * np.exp(-r * T) * norm.cdf(-d2)

  else:
      raise ValueError("option_type must be 'call' or 'put'")


  # --- 3. Shared Greeks Calculation ---
  # Gamma and Vega are identical for both Call and Put options,
  # so they are calculated outside the conditional blocks.
  gamma = norm.pdf(d1) / (S * sigma * np.sqrt(T))
  vega = S * np.sqrt(T) * norm.pdf(d1)

  return delta, gamma, vega, theta, rho

In [ ]:
def price_report_with_vanilla(S, K, T, r, sigma, option_type):
    """
    Generates a comparative pricing report for a vanilla European option
    across three different quantitative models: Black-Scholes, Monte Carlo,
    and the Binomial Tree.

    Outputs a formatted Pandas DataFrame to the display environment.

    Parameters:
    -----------
    S : float
        Current spot price of the underlying asset.
    K : float
        Strike price of the option contract.
    T : float
        Time to maturity in years.
    r : float
        Annualized continuously compounded risk-free interest rate.
    sigma : float
        Annualized volatility of the underlying asset's returns.
    option_type : str
        The type of option contract: "call" or "put".

    Dependencies:
    -------------
    Requires `pandas` as `pd` and a Jupyter notebook/IPython environment for `display()`.
    """
    # 1. Execute pricing routines across all engines
    price_bs = black_scholes_price(S, K, T, r, sigma, option_type)
    price_mc = monte_carlo_price(S, K, T, r, sigma, option_type)
    price_bt = binomial_tree_price(S, K, T, r, sigma, option_type)

    # 2. Consolidate premiums into a comparative DataFrame
    model = ["Black-Scholes", "Monte Carlo", "Binomial Tree"]
    price_table = pd.DataFrame({
        "Price": [price_bs, price_mc, price_bt]
    }, index=model).round(2)

    # 3. Output results
    print(f"The Price of {option_type.capitalize()} option")
    display(price_table)

In [ ]:
def greek_report_with_vanilla(S, K, T, r, sigma, option_type):
    """
    Generates a comprehensive benchmarking report comparing option Greeks
    calculated analytically via Closed-Form solutions against numerical
    Finite Difference Method (FDM) implementations.

    CRITICAL NOTE FOR CODE REVIEWERS:
    ---------------------------------
    - The 'Monte Carlo (FDM)' outputs will exhibit massive variance/noise
      unless the underlying engine locks paths using Common Random Numbers (CRN).
    - The 'Binomial Tree' row reflects FDM shocks on a single-period model,
      causing visible tracking errors against the analytical benchmark.

    Parameters:
    -----------
    S : float
        Current spot price of the underlying asset.
    K : float
        Strike price of the option contract.
    T : float
        Time to maturity in years.
    r : float
        Annualized continuously compounded risk-free interest rate.
    sigma : float
        Annualized volatility of the underlying asset's returns.
    option_type : str
        The type of option contract: "call" or "put".
    """
    # 1. Extract exact analytical Greeks from the Closed-Form Black-Scholes solution
    delta_cf, gamma_cf, vega_cf, theta_cf, rho_cf = greeks_fdm_cf(S, K, T, r, sigma, option_type)

    # 2. Extract numerical Greeks via analytical Black-Scholes FDM
    delta_fdm_bs, gamma_fdm_bs, vega_fdm_bs, theta_fdm_bs, rho_fdm_bs = greeks_fdm(black_scholes_price, S, K, T, r, sigma, option_type)

    # 3. Extract numerical Greeks via Monte Carlo Simulation FDM
    delta_fdm_mc, gamma_fdm_mc, vega_fdm_mc, theta_fdm_mc, rho_fdm_mc = greeks_fdm(monte_carlo_price, S, K, T, r, sigma, option_type)

    # 4. Extract numerical Greeks via Binomial Tree FDM
    delta_fdm_bt, gamma_fdm_bt, vega_fdm_bt, theta_fdm_bt, rho_fdm_bt = greeks_fdm(binomial_tree_price, S, K, T, r, sigma, option_type)

    # 5. Build structured comparative matrix using Pandas
    methods = ["Closed-Form (Analytical)", "FDM (Black-Scholes)", "Monte Carlo (FDM)", "Binomial Tree"]
    greek_table = pd.DataFrame({
        "Delta": [delta_cf, delta_fdm_bs, delta_fdm_mc, delta_fdm_bt],
        "Gamma": [gamma_cf, gamma_fdm_bs, gamma_fdm_mc, gamma_fdm_bt],
        "Vega":  [vega_cf,  vega_fdm_bs,  vega_fdm_mc,  vega_fdm_bt],
        "Theta": [theta_cf, theta_fdm_bs, theta_fdm_mc, theta_fdm_bt],
        "Rho":   [rho_cf,   rho_fdm_bs,   rho_fdm_mc,   rho_fdm_bt],
    }, index=methods).round(6)

    # 6. Output formatted analysis report
    print(f"The Greeks of {option_type.capitalize()} option")
    print(greek_table)
    print("-" * 80)

In [ ]:
def _bond_and_budget(notional, PR, r, T):
    """Shared bond-leg / option-budget split, with the solvency check the
    original code was missing."""
    bond_leg = notional * PR * np.exp(-r * T)
    option_budget = notional - bond_leg

    return bond_leg, option_budget

In [ ]:
def structuring_note_with_vanilla(notional, PR, S, K, T, r, sigma, option_type):
  bond_leg, option_budget = _bond_and_budget(notional, PR, r, T)
  option_price = black_scholes_price(S, K, T, r, sigma, option_type)
  uints_of_options = option_budget/option_price
  return {
        "Bond Value": bond_leg,
        "Option Budget": option_budget,
        "Fixed PR": PR,
        "Option Price": option_price,
        "Units of option": uints_of_options,
        "Time" : T
    }

In [ ]:
def structuring_note_bull_call_spread(notional, PR, S, K_low, K_high, T, r, sigma):

  if K_high > K_low:
    bond_leg, option_budget = _bond_and_budget(notional, PR, r, T)

    call_K_low = black_scholes_price(S, K_low, T, r, sigma, "call")
    call_K_high = black_scholes_price(S, K_high, T, r, sigma, "call")
    option_price = call_K_low - call_K_high

    uints_of_options = option_budget/option_price

  else:
    raise ValueError("Exercise price in this function must be K_high > K_low")

  return {
        "Bond Value": bond_leg,
        "Option Budget": option_budget,
        "Fixed PR": PR,
        "Option Price": option_price,
        "Units of option": uints_of_options,
        "Time" : T
    }

In [ ]:
def structuring_note_straddle(notional, PR, S, K, T, r, sigma):
  bond_leg, option_budget = _bond_and_budget(notional, PR, r, T)
  call = black_scholes_price(S, K, T, r, sigma, "call")
  put = black_scholes_price(S, K, T, r, sigma, "put")
  option_price = call + put
  uints_of_options = option_budget/option_price
  return {
        "Bond Value": bond_leg,
        "Option Budget": option_budget,
        "Fixed PR": PR,
        "Option Price": option_price,
        "Units of option": uints_of_options,
        "Time" : T
    }

In [ ]:
def structuring_note_bull_strangle(notional, PR, S, K_call, K_put, T, r, sigma):

  if K_put > K_call:
    bond_leg, option_budget = _bond_and_budget(notional, PR, r, T)

    call = black_scholes_price(S, K_call, T, r, sigma, "call")
    put  = black_scholes_price(S, K_put, T, r, sigma, "put")
    option_price = call - put

    uints_of_options = option_budget/option_price

  else:
    raise ValueError("Exercise price in this function must be K_high > K_low")

  return {
        "Bond Value": bond_leg,
        "Option Budget": option_budget,
        "Fixed PR": PR,
        "Option Price": option_price,
        "Units of option": uints_of_options,
        "Time" : T
    }

In [ ]:
def print_note_summary(title: str, note_data: dict):
    print(title)
    # ใช้การวนลูปผ่าน key ของ Dictionary แทนการเขียนระบุทีละตัว
    # กำหนดความกว้างของตัวอักษรเป็น 16 ตัวอักษร (:<16) เพื่อให้เครื่องหมาย : ตรงกัน
    for key, value in note_data.items():
        print(f"{key:<16}: {value:.2f}")
    print("-" * 40)

In [ ]:
# --- Core Inputs (Standard Annualized Basis) ---
notional = 10000
PR = 0.8
S = 100
K = 99
T = 10
r = 0.015/12
sigma = 0.32/np.sqrt(12)
option_type = "call"

In [ ]:
for opt in ["call","put"]:
  greek_report_with_vanilla(S, K, T, r, sigma, opt)

In [ ]:
# 1. Vanilla Options (Call & Put)
for opt in ["call", "put"]:
    data = structuring_note_with_vanilla(notional, PR, S, K, T, r, sigma, opt)
    print_note_summary(f"Structuring note with {opt.capitalize()} Option", data)

# 2. Bull Call Spread
s1_data = structuring_note_bull_call_spread(notional, PR, S, S * 0.90, S * 1.10, T, r, sigma)
print_note_summary("Structuring note bull call spread", s1_data)

# 3. Straddle
s2_data = structuring_note_straddle(notional, PR, S, K, T, r, sigma)
print_note_summary("Structuring note straddle", s2_data)

# 4. Bull Strangle
s3_data = structuring_note_bull_strangle(notional, PR, S, S * 0.90, S * 1.10, T, r, sigma)
print_note_summary("Structuring note bull strangle", s3_data)